# 03 -- DRW Parameter Estimation Pipeline

For each quasar in the Stripe 82 catalog we:
1. Cone search in **ZTF DR23** (AWS S3 via `lsdb`) for r-band light curves
2. Select the closest counterpart by angular separation
3. Clean the light curve (catflags + magerr cut)
4. Fit a **Damped Random Walk (DRW)** model with `celerite` + `emcee` MCMC
5. Extract tau (timescale), sigma (amplitude), mu (mean magnitude)
6. Save results as features for the black-hole mass prediction step

**Input:** `data/DR16Q_final_stripe82.fits`  
**Outputs:** `data/DRW_results.fits`, `data/DRW_results.csv`

### DRW kernel recap
In `celerite` a DRW is a single `RealTerm` with:
```
log_a = 2*log(sigma)  =>  sigma = sqrt(exp(log_a)/2)  [variability amplitude, mag]
log_c = -log(tau)     =>  tau   = exp(-log_c)           [damping timescale, days]
```
The mean magnitude `mu` is a free MCMC parameter.

In [ ]:
## Install required packages (uncomment on first run)
# !pip install astropy pandas "numpy<2.0" emcee celerite
# !pip install "lsdb==0.9.0" hats s3fs

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

import lsdb
import celerite
from celerite import terms
import emcee

DATA_DIR   = Path('data')
OUTPUT_DIR = DATA_DIR

## 1. Load the Stripe 82 Quasar Catalog

In [ ]:
DATA_PATH = DATA_DIR / 'DR16Q_final_stripe82.fits'
t_qso = Table.read(DATA_PATH)

row = t_qso[0]
print(f'Loaded {len(t_qso):,} quasars')
print(f'Columns: {t_qso.colnames}')
print(f'Example: {row["SDSS_NAME"]}, RA={float(row["RA"]):.6f}, Dec={float(row["DEC"]):.6f}')

## 2. ZTF DR23 Light Curve Fetching

We use **lsdb** to read ZTF DR23 from AWS S3 in HiPSCat/HATS (spatially partitioned Parquet) format.
A cone search loads only the sky partitions that overlap our target -- no full dataset download.

**Filter ID mapping:** `1=g`, `2=r`, `3=i`. We use r-band only (`filterid==2`).

In [ ]:
ZTF_DR23_URL = "s3://ipac-irsa-ztf/contributed/dr23/lc/hats"


def fetch_ztf_lightcurves_lsdb(ra, dec, radius_arcsec=2.0):
    # Cone search in ZTF DR23. Returns list of DataFrames (one per source/filter).
    # Each DataFrame has catflags-cleaned rows and source metadata columns.
    cat = lsdb.open_catalog(
        ZTF_DR23_URL,
        search_filter=lsdb.ConeSearch(ra, dec, radius_arcsec=radius_arcsec),
    )
    df = cat.compute()

    lightcurves = []
    for _, src in df.iterrows():
        lc = src.get("lightcurve", None)
        if lc is None or len(lc) == 0:
            continue
        lc = lc.copy()
        if "catflags" in lc.columns:
            lc = lc[lc["catflags"] == 0].copy()
        if len(lc) == 0:
            continue
        lc["objectid"] = src.get("objectid", np.nan)
        lc["filterid"] = src.get("filterid", np.nan)
        lc["objra"]    = src.get("objra",    np.nan)
        lc["objdec"]   = src.get("objdec",   np.nan)
        lightcurves.append(lc)
    return lightcurves


def select_best_r_lightcurve(lightcurves, ra, dec):
    # From all cone-search results, return the r-band source (filterid==2)
    # with the smallest angular separation from (ra, dec).
    if not lightcurves:
        return None
    r_band = [lc for lc in lightcurves
              if "filterid" in lc.columns and int(lc["filterid"].iloc[0]) == 2]
    if not r_band:
        return None

    ref = SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
    best_lc, best_sep = None, None
    for lc in r_band:
        if "objra" not in lc.columns or "objdec" not in lc.columns:
            continue
        src_coord = SkyCoord(
            ra=float(lc["objra"].iloc[0]) * u.deg,
            dec=float(lc["objdec"].iloc[0]) * u.deg,
        )
        sep = ref.separation(src_coord)
        if best_sep is None or sep < best_sep:
            best_sep = sep
            best_lc  = lc

    if best_lc is not None:
        print(f"  Best r-band match: OID={best_lc['objectid'].iloc[0]}, "
              f"separation={best_sep.arcsec:.3f} arcsec")
    return best_lc

## 3. Light Curve Cleaning

Apply magerr < 0.2 mag cut and return time-sorted arrays.
The catflags mask was applied at fetch time but we re-check defensively.

In [ ]:
def clean_r_band(lc):
    # Apply quality cuts; return (t, y, yerr) arrays sorted by time.
    if "filterid" in lc.columns:
        lc = lc[lc["filterid"] == 2]
    if "catflags" in lc.columns:
        lc = lc[lc["catflags"] == 0]
    if "magerr" in lc.columns:
        lc = lc[lc["magerr"] < 0.2]

    t    = np.array(lc["hmjd"])
    y    = np.array(lc["mag"])
    yerr = np.array(lc["magerr"])
    idx  = np.argsort(t)
    return t[idx], y[idx], yerr[idx]

## 4. DRW Model -- celerite GP

We model quasar variability as a Damped Random Walk (first-order Ornstein-Uhlenbeck process).
In `celerite` this is a `RealTerm` kernel.

In [ ]:
def build_gp(t, yerr):
    # Initialise celerite GP with DRW kernel.
    # Starting values: sigma~0.1 mag, tau~100 days
    log_sigma = np.log(0.1)
    log_tau   = np.log(100.0)
    kernel = terms.RealTerm(log_a=2.0 * log_sigma, log_c=-log_tau)
    gp = celerite.GP(kernel)
    gp.compute(t, yerr)
    return gp

## 5. MCMC -- emcee

Three free parameters: `log_a` (amplitude), `log_c` (timescale), `mu` (mean magnitude).
Flat prior with physically motivated bounds.

In [ ]:
def log_prior(theta):
    log_a, log_c, mu = theta
    if -10 < log_a < 5 and -10 < log_c < 10 and 10 < mu < 25:
        return 0.0     # flat prior within bounds
    return -np.inf


def log_likelihood(theta, gp, y):
    log_a, log_c, mu = theta
    gp.set_parameter_vector([log_a, log_c])
    return gp.log_likelihood(y - mu)    # subtract mu so GP sees zero-mean residuals


def log_probability(theta, gp, y):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = log_likelihood(theta, gp, y)
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


def run_mcmc(t, y, yerr, nwalkers=32, nsteps=2000, discard=500):
    # Run emcee and return flattened posterior samples after burn-in.
    gp      = build_gp(t, yerr)
    initial = np.array([np.log(0.1), np.log(100.0), np.mean(y)])
    ndim    = len(initial)
    p0      = initial + 1e-4 * np.random.randn(nwalkers, ndim)

    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=(gp, y))
    sampler.run_mcmc(p0, nsteps, progress=True)
    return sampler.get_chain(discard=discard, flat=True)

## 6. Extract DRW Parameters from Posterior

In [ ]:
def extract_params(samples):
    # Return (tau, sigma, mu) from the MCMC posterior median.
    log_a_med, log_c_med, mu_med = np.median(samples, axis=0)
    tau   = np.exp(-log_c_med)
    sigma = np.sqrt(np.exp(log_a_med) / 2.0)
    return tau, sigma, mu_med

## 7. Run the Full Pipeline

Iterates over all Stripe 82 quasars and collects DRW parameters.

**Tip:** Test on a small slice first (e.g. `t_qso[:4]`) before processing the full catalog.

In [ ]:
MIN_EPOCHS = 10   # minimum clean r-band epochs for a meaningful DRW fit

results = []

# Replace t_qso[:4] with t_qso to process the full catalog
for row in t_qso[:4]:
    sdss_name = row["SDSS_NAME"]
    ra, dec   = float(row["RA"]), float(row["DEC"])
    logmbh    = float(row["LOGMBH"]) if "LOGMBH" in t_qso.colnames else np.nan
    psfmag    = row["PSFMAG"] if "PSFMAG" in t_qso.colnames else np.nan
    if hasattr(psfmag, "tolist"):
        psfmag = psfmag.tolist()

    print(f"\n--- {sdss_name}  (RA={ra:.5f}, Dec={dec:.5f}) ---")

    # Step 1: cone search
    try:
        lcs = fetch_ztf_lightcurves_lsdb(ra, dec, radius_arcsec=2.0)
    except Exception as e:
        print(f"  [ERROR] lsdb fetch failed: {e}")
        continue

    if not lcs:
        print("  No ZTF sources found -- skipping")
        continue

    # Step 2: select closest r-band source
    lc_best = select_best_r_lightcurve(lcs, ra, dec)
    if lc_best is None:
        print("  No r-band data for closest match -- skipping")
        continue

    oid       = lc_best["objectid"].iloc[0]
    match_ra  = float(lc_best["objra"].iloc[0])  if "objra"  in lc_best.columns else np.nan
    match_dec = float(lc_best["objdec"].iloc[0]) if "objdec" in lc_best.columns else np.nan

    if np.isfinite(match_ra) and np.isfinite(match_dec):
        sep_arcsec = (
            SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
            .separation(SkyCoord(ra=match_ra * u.deg, dec=match_dec * u.deg))
            .arcsec
        )
    else:
        sep_arcsec = np.nan

    # Step 3: clean the light curve
    t_lc, y_lc, yerr_lc = clean_r_band(lc_best)
    if len(t_lc) < MIN_EPOCHS:
        print(f"  Only {len(t_lc)} clean epochs (need {MIN_EPOCHS}) -- skipping")
        continue

    print(f"  {len(t_lc)} clean r-band epochs -- running MCMC ...")

    # Step 4: DRW fit
    try:
        samples = run_mcmc(t_lc, y_lc, yerr_lc)
    except Exception as e:
        print(f"  [ERROR] MCMC failed: {e}")
        continue

    # Step 5: extract parameters
    tau, sigma, mu = extract_params(samples)
    print(f"  tau={tau:.1f} d,  sigma={sigma:.4f} mag,  mu={mu:.3f} mag")

    results.append({
        "SDSS_NAME"        : sdss_name,
        "target_ra"        : ra,
        "target_dec"       : dec,
        "match_ra"         : match_ra,
        "match_dec"        : match_dec,
        "separation_arcsec": sep_arcsec,
        "LOGMBH"           : logmbh,
        "PSFMAG"           : psfmag,
        "objectid"         : oid,
        "tau"              : tau,
        "sigma"            : sigma,
        "mu"               : mu,
        "n_epochs"         : len(t_lc),
    })

print(f"\n=== Done: {len(results)} quasars processed ===")

## 8. Save Results

In [ ]:
from astropy.table import Table as AstropyTable

if results:
    df_results = pd.DataFrame(results)

    out_fits = OUTPUT_DIR / "DRW_results.fits"
    AstropyTable.from_pandas(df_results).write(str(out_fits), overwrite=True)
    print(f"Saved {len(df_results)} rows to {out_fits}")

    out_csv = OUTPUT_DIR / "DRW_results.csv"
    df_results.to_csv(out_csv, index=False)
    print(f"Saved CSV to {out_csv}")

    print("\nPreview:")
    print(df_results[["SDSS_NAME", "tau", "sigma", "mu", "n_epochs", "LOGMBH"]].head())
else:
    print("No results to save.")